
# Evaluating the Effectiveness of Pre-trained Models from Different Machine Learning Model Hubs

Here I explore and evaluate pre-trained models from Kaggle, PyTorch Hub, TensorFlow Hub, and Hugging Face. I consider the performance, computational resources, and suitability for specific tasks.

### Imports


In [3]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install -U "tf-keras==2.16.0"

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import gc
import warnings
warnings.filterwarnings("ignore")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # quiet OpenMP duplicates
os.environ["OMP_NUM_THREADS"] = "1"          # reduce thread contention
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
#----- std lib -----
import re
import random
import numpy as np
import time
import pandas as pd
# ----- data loading (HF datasets) -----
from datasets import load_dataset
# ----- PyTorch stack -----
import torch
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torch.utils.data import ConcatDataset
import torchvision
import torchvision.datasets as datasets
from datasets import DatasetDict
from datasets import concatenate_datasets
import torchvision.transforms as T   # use ONE alias; avoid importing transforms twice
from torch.optim import AdamW
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn
# ----- Baseline (scikit-learn) -----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV

In [10]:
# ----- Hugging Face Transformers -----
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

In [11]:
# ----- TensorFlow stack -----
import tensorflow as tf 
import tensorflow_datasets as tfds 
import tensorflow_hub as hub
import tf_keras as keras 

### Loading Datasets

In [13]:
# Load IMDb reviews 
imdb = load_dataset("imdb")

# Peek
print(imdb)
print(imdb["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [14]:
# Raw CIFAR-10 (32x32 images)
cifar_train = datasets.CIFAR10(root="./data", train=True, download=True)
cifar_test = datasets.CIFAR10(root="./data", train=False, download=True)

print(len(cifar_train), len(cifar_test))
print(cifar_train[0])  # (image, label)

100%|███████████████████████| 170498071/170498071 [00:06<00:00, 25805366.47it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
50000 10000
(<PIL.Image.Image image mode=RGB size=32x32 at 0x3403F6680>, 6)


#### Base Preprocessing for IMBD dataset. Keeps text + label, stripping metadata; lowercases all text & removes HTML tags & odd chars.

In [16]:
# Simple cleaning function
def clean_text(text):
    text = text.lower()  # standardize casing
    text = re.sub(r"<br\s*/?>", " ", text)  # remove HTML line breaks
    text = re.sub(r"[^a-z0-9\s]", "", text)  # remove punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()  # normalize whitespace
    return text

# Apply cleaning
imdb = imdb.map(lambda x: {"text": clean_text(x["text"])}, batched=False)

#### Base Preprocessing for CIFAR-10 dataset. Converts to tensors; normalizes pixel values to [0,1].

In [18]:
# Base transform: just convert to tensor + scale pixels to [0,1]
base_transform = T.ToTensor()

# Load CIFAR-10 train/test
cifar_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=base_transform)
cifar_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=base_transform)

# Check shape and value range
img, label = cifar_train[0]
print(img.shape, img.min().item(), img.max().item(), label)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([3, 32, 32]) 0.0 1.0 6


### Model-Specific Preprocessing

#### NLP - DistilBERT (HuggingFace) Preprocessing. DistilBERT’s tokenizer defines the input shape/IDs, so load it first, then map over IMDb, then load the model.

In [21]:
# --- DistilBERT preprocess → PyTorch DataLoaders (no aug by design) ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_ID = "distilbert-base-uncased"
MAX_LEN  = 128
BATCH    = 32

# If imdb has an 'unsupervised' split, drop it
if hasattr(imdb, "keys"):
    imdb = DatasetDict({k: imdb[k] for k in imdb.keys() if k in ("train", "test")})

tok = AutoTokenizer.from_pretrained(MODEL_ID)

def tok_fn(batch):
    return tok(batch["text"], padding=False, truncation=True, max_length=MAX_LEN)

imdb_tok = imdb.map(tok_fn, batched=True, remove_columns=[c for c in imdb["train"].column_names if c not in ("label",)])
if "label" in imdb_tok["train"].column_names:
    imdb_tok = imdb_tok.rename_column("label", "labels")

cols = [c for c in ["input_ids","attention_mask","token_type_ids","labels"] if c in imdb_tok["train"].column_names]
imdb_tok.set_format(type="torch", columns=cols)

collator = DataCollatorWithPadding(tokenizer=tok)
hf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

train_loader_hf = DataLoader(imdb_tok["train"], batch_size=BATCH, shuffle=True,  collate_fn=collator, num_workers=0)
test_loader_hf  = DataLoader(imdb_tok["test"],  batch_size=BATCH, shuffle=False, collate_fn=collator, num_workers=0)

# Preprocessing description for results table
PREPROC_DESC_HF_NLP = "Lowercase+HTML/punct cleanup; HF tokenizer (uncased), trunc=128, dynamic padding; no data aug"

# sanity check
batch = next(iter(train_loader_hf))
print({k: v.shape for k,v in batch.items()})

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'labels': torch.Size([32]), 'input_ids': torch.Size([32, 128]), 'attention_mask': torch.Size([32, 128])}


#### CV - ResNet18 (PyTorch Hub). ResNet18 is ImageNet-pretrained; it expects 224×224 and ImageNet mean/std. Define transforms first, then wrap CIFAR-10, then load the model.

In [23]:
# --- ResNet18 preprocess → PyTorch DataLoaders (NO augmentation) ---
IMGNET_MEAN = [0.485, 0.456, 0.406]
IMGNET_STD  = [0.229, 0.224, 0.225]

# Resize to 256, then center-crop to 224, normalize with ImageNet stats — no flips/augs
common_tfms_resnet = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(IMGNET_MEAN, IMGNET_STD),
])

cifar_train_resnet = datasets.CIFAR10(root="./data", train=True,  download=True, transform=common_tfms_resnet)
cifar_test_resnet  = datasets.CIFAR10(root="./data", train=False, download=True, transform=common_tfms_resnet)

train_loader_resnet = DataLoader(cifar_train_resnet, batch_size=64, shuffle=True,  num_workers=0)
test_loader_resnet  = DataLoader(cifar_test_resnet,  batch_size=64, shuffle=False, num_workers=0)

PREPROC_DESC_RESNET = "Resize256→CenterCrop224; ToTensor; Normalize(ImageNet mean/std); no aug"

# sanity check
xb, yb = next(iter(train_loader_resnet))
print(xb.shape, yb.shape, float(xb.amin()), float(xb.amax()))

Files already downloaded and verified
Files already downloaded and verified
torch.Size([64, 3, 224, 224]) torch.Size([64]) -2.1179039478302 2.640000104904175


#### CV — MobileNetV2 (TensorFlow Hub) TF-Hub MobileNetV2 feature vectors expect 224×224 and float32 in [0,1] (feature_vector variants don’t need ImageNet mean/std). Define a tf.data pipeline first, then build the model.

In [25]:
# --- MobileNetV2 (TF-Hub) preprocess → tf.data (NO augmentation, tf_keras) ---
IMG_SIZE = 224
BATCH    = 64

def tf_preprocess_no_aug(image, label):
    # Match PyTorch path: Resize to 256 then center-crop to 224, then scale to [0,1]
    image = tf.image.resize(image, (256, 256), method="bilinear")
    offset = (256 - IMG_SIZE) // 2  # 16
    image = tf.image.crop_to_bounding_box(image, offset, offset, IMG_SIZE, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds, test_ds = tfds.load("cifar10", split=["train", "test"], as_supervised=True, shuffle_files=True)
train_ds = (train_ds.map(tf_preprocess_no_aug, num_parallel_calls=tf.data.AUTOTUNE)
                    .shuffle(10_000).batch(BATCH).prefetch(tf.data.AUTOTUNE))
test_ds  = (test_ds.map(tf_preprocess_no_aug,  num_parallel_calls=tf.data.AUTOTUNE)
                   .batch(BATCH).prefetch(tf.data.AUTOTUNE))

# TF-Hub feature vector + small classification head (Functional API w/ tf_keras)
FV_URL = "https://tfhub.dev/google/imagenet/mobilenet_v2_130_224/feature_vector/5"
feat   = hub.KerasLayer(FV_URL, input_shape=(IMG_SIZE, IMG_SIZE, 3), trainable=False)

inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = feat(inputs)  # feature vector (e.g., 1664 dims for 1.3x)
outputs = keras.layers.Dense(10, activation="softmax")(x)
model   = keras.Model(inputs, outputs)

# (optional) compile now so it's ready to train
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

PREPROC_DESC_TF_MBV2 = "Resize256→CenterCrop224; scale to [0,1]; feature_vector; no aug"

# sanity check
x0, y0 = next(iter(train_ds))
print(x0.shape, y0.shape, x0.dtype, float(tf.reduce_min(x0)), float(tf.reduce_max(x0)))
yhat = model(x0[:2])
print("logits shape:", yhat.shape)  # expect (2, 10)

2025-09-12 01:48:47.761702: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-09-12 01:48:47.761729: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-09-12 01:48:47.761736: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-09-12 01:48:47.761769: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-12 01:48:47.761780: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-09-12 01:48:49.141403: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected tr

(64, 224, 224, 3) (64,) <dtype: 'float32'> 0.0 1.0


2025-09-12 01:48:49.499612: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


logits shape: (2, 10)


#### NLP — Kaggle-style lightweight baseline (scikit-learn) Linear models don’t use token IDs; define vectorization first, then fit the model.

In [27]:
# --- TF-IDF + Logistic Regression baseline (NO augmentation, deterministic text clean already done) ---
X_train = imdb["train"]["text"]; y_train = imdb["train"]["labels"] if "labels" in imdb["train"].column_names else imdb["train"]["label"]
X_test  = imdb["test"]["text"];  y_test  = imdb["test"]["labels"]  if "labels"  in imdb["test"].column_names  else imdb["test"]["label"]

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=40_000, ngram_range=(1,2))),
    ("clf",   LogisticRegression(max_iter=1000, n_jobs=-1))
])

# fit once as a sanity check (training proper can be elsewhere)
tfidf_lr.fit(X_train, y_train)
print("Baseline accuracy:", tfidf_lr.score(X_test, y_test))

PREPROC_DESC_TFIDF = "Lowercase+HTML/punct cleanup; TF-IDF(max_features=40k, 1–2gram); no aug"

Baseline accuracy: 0.89648


### Linear-probe stage (freeze backbones, train heads only) for each model, then log metrics & preprocessing string.

#### DistilBERT — linear probe (3 epochs, head only)

In [30]:
device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
hf_model.to(device)

# freeze backbone
for p in hf_model.distilbert.parameters():
    p.requires_grad = False

opt = AdamW(
    list(hf_model.pre_classifier.parameters()) + list(hf_model.classifier.parameters()),
    lr=2e-4
)

def eval_hf(loader):
    hf_model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k,v in batch.items()}
            out = hf_model(**batch)
            ps.append(out.logits.argmax(-1).cpu().numpy())
            ys.append(batch["labels"].cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p, average="macro")

EPOCHS = 3
t0 = time.time()
for e in range(1, EPOCHS+1):
    hf_model.train()
    running = 0.0
    for batch in train_loader_hf:
        batch = {k: v.to(device) for k,v in batch.items()}
        out = hf_model(**batch)
        loss = out.loss
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
    acc, f1 = eval_hf(test_loader_hf)
    print(f"[DistilBERT] Epoch {e}  loss={running/len(train_loader_hf):.4f}  acc={acc:.4f}  f1={f1:.4f}")
train_time_min = (time.time()-t0)/60

# log (uses the helper from earlier; create if missing)
try:
    log_result
except NameError:
    import pandas as pd
    results = []
    def log_result(**kw): results.append(kw)

log_result(hub="Hugging Face", model="DistilBERT-base-uncased", task="IMDb",
           f1=f1, acc=acc, train_min=train_time_min, preprocessing=PREPROC_DESC_HF_NLP, notes="linear-probe")

[DistilBERT] Epoch 1  loss=0.4718  acc=0.7984  f1=0.7973
[DistilBERT] Epoch 2  loss=0.4325  acc=0.7949  f1=0.7926
[DistilBERT] Epoch 3  loss=0.4235  acc=0.8165  f1=0.8164


#### ResNet18 — linear probe (replace FC, freeze backbone)

In [32]:
device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

# model + head
r18 = resnet18(weights=ResNet18_Weights.DEFAULT)
in_features = r18.fc.in_features
r18.fc = nn.Linear(in_features, 10)

# freeze backbone
for name, p in r18.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

r18 = r18.to(device)
opt = AdamW(r18.fc.parameters(), lr=3e-4)

def eval_cv(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            ps.append(logits.argmax(1).cpu().numpy())
            ys.append(yb.cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p, average="macro")

EPOCHS = 3
t0 = time.time()
for e in range(1, EPOCHS+1):
    r18.train(); running = 0.0
    for xb, yb in train_loader_resnet:
        xb = xb.to(device); yb = yb.to(device)
        logits = r18(xb)
        loss = nn.functional.cross_entropy(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
    acc_r, f1_r = eval_cv(r18, test_loader_resnet)
    print(f"[ResNet18] Epoch {e}  loss={running/len(train_loader_resnet):.4f}  acc={acc_r:.4f}  f1={f1_r:.4f}")
train_time_min_r = (time.time()-t0)/60

log_result(hub="PyTorch Hub", model="ResNet18", task="CIFAR-10",
           f1=f1_r, acc=acc_r, train_min=train_time_min_r, preprocessing=PREPROC_DESC_RESNET, notes="linear-probe")

[ResNet18] Epoch 1  loss=1.1861  acc=0.7342  f1=0.7324
[ResNet18] Epoch 2  loss=0.7943  acc=0.7590  f1=0.7577
[ResNet18] Epoch 3  loss=0.7268  acc=0.7668  f1=0.7657


#### TF-Hub MobileNetV2 — linear probe (feature vector + dense head)

In [34]:
t0 = time.time()
hist = model.fit(train_ds, validation_data=test_ds, epochs=3, verbose=1)
train_time_min_tf = (time.time()-t0)/60

# evaluate with sklearn F1 for consistency
ys, ps = [], []
for xb, yb in test_ds:
    yhat = model.predict(xb, verbose=0)
    ps.append(np.argmax(yhat, axis=1))
    ys.append(yb.numpy())
y = np.concatenate(ys); p = np.concatenate(ps)
acc_tf = accuracy_score(y,p); f1_tf = f1_score(y,p, average="macro")
print(f"[TF Hub MBv2] acc={acc_tf:.4f}  f1={f1_tf:.4f}")

log_result(hub="TF Hub", model="MobileNetV2-130 (feature_vector)", task="CIFAR-10",
           f1=f1_tf, acc=acc_tf, train_min=train_time_min_tf, preprocessing=PREPROC_DESC_TF_MBV2, notes="linear-probe")

Epoch 1/3


2025-09-12 02:02:20.889724: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


782/782 [==============================] - 87s 102ms/step - loss: 0.5274 - accuracy: 0.8188 - val_loss: 0.4214 - val_accuracy: 0.8520
Epoch 2/3
782/782 [==============================] - 81s 102ms/step - loss: 0.3722 - accuracy: 0.8706 - val_loss: 0.3991 - val_accuracy: 0.8621
Epoch 3/3
782/782 [==============================] - 84s 107ms/step - loss: 0.3368 - accuracy: 0.8826 - val_loss: 0.4118 - val_accuracy: 0.8567
[TF Hub MBv2] acc=0.8567  f1=0.8587


2025-09-12 02:06:43.204157: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


#### TF-IDF + Logistic Regression — already trained

In [36]:
y_pred = tfidf_lr.predict(X_test)
f1_base = f1_score(y_test, y_pred, average="macro")
acc_base = (y_pred == y_test).mean()
log_result(hub="Kaggle-style", model="TF-IDF + LogisticRegression", task="IMDb",
           f1=f1_base, acc=acc_base, train_min=None, preprocessing=PREPROC_DESC_TFIDF, notes="baseline")

#### View and save results

In [38]:
df_results = pd.DataFrame(results)
display(df_results)
df_results.to_csv("light_models_linear_probe_results.csv", index=False)

,hub,model,task,f1,acc,train_min,preprocessing,notes
0,Hugging Face,DistilBERT-base-uncased,IMDb,0.816447,0.81652,8.613636,Lowercase+HTML/punct cleanup; HF tokenizer (un...,linear-probe
1,PyTorch Hub,ResNet18,CIFAR-10,0.765713,0.76680,4.653751,Resize256→CenterCrop224; ToTensor; Normalize(I...,linear-probe
2,TF Hub,MobileNetV2-130 (feature_vector),CIFAR-10,0.858679,0.85670,4.186346,"Resize256→CenterCrop224; scale to [0,1]; featu...",linear-probe
3,Kaggle-style,TF-IDF + LogisticRegression,IMDb,0.896477,0.89648,NaN,Lowercase+HTML/punct cleanup; TF-IDF(max_featu...,baseline


### Optimization Stage - Now we will fold in grid search optimization toe hfinal layer of these models to try and derive better accuracies. These grid searches vary slightly between models.

#### Validation Splits for optimization

In [41]:
# IMDb: 90/10 train/val
imdb_train_val = imdb_tok["train"].train_test_split(test_size=0.1, seed=42)
imdb_train_ds, imdb_val_ds = imdb_train_val["train"], imdb_train_val["test"]

# CIFAR-10 (PyTorch path): 45k/5k split from train
val_size = 5000
train_size = len(cifar_train_resnet) - val_size
cifar_train_resnet_split, cifar_val_resnet = random_split(
    cifar_train_resnet, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_loader_resnet_s = DataLoader(cifar_train_resnet_split, batch_size=64, shuffle=True,  num_workers=0)
val_loader_resnet     = DataLoader(cifar_val_resnet,        batch_size=64, shuffle=False, num_workers=0)

# CIFAR-10 (TF path): take 5k from train for val
val_ds = train_ds.take(5000 // 64)  # ~5k images (batch 64)
train_ds_s = train_ds.skip(5000 // 64)

#### DistilBERT — light fine-tune + tiny search. Unfreeze: last transformer layer only; head LR: {1e-4, 2e-4}; backbone LR: {3e-5, 5e-5}; weight decay: {0, 0.01}; epochs: 3

In [43]:
device = torch.device("mps") if torch.backends.mps.is_available() else \
         torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
hf_model.to(device)

def eval_loader(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) for k,v in b.items()}
            out = model(**b)
            ps.append(out.logits.argmax(-1).cpu().numpy())
            ys.append(b["labels"].cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p, average="macro")

def run_ft(head_lr, bb_lr, weight_decay, epochs=3):
    # fresh model each trial (head randomly init)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2).to(device)

    # freeze all, then unfreeze last transformer layer
    for p in model.distilbert.parameters(): p.requires_grad = False
    for p in model.distilbert.transformer.layer[-1].parameters(): p.requires_grad = True

    # parameter groups: head vs last layer
    head_params = list(model.pre_classifier.parameters()) + list(model.classifier.parameters())
    last_layer_params = list(model.distilbert.transformer.layer[-1].parameters())

    opt = AdamW([
        {"params": head_params, "lr": head_lr, "weight_decay": weight_decay},
        {"params": last_layer_params, "lr": bb_lr, "weight_decay": weight_decay},
    ])

    # loaders for this split
    collator = DataCollatorWithPadding(tokenizer=tok)
    train_loader = DataLoader(imdb_train_ds, batch_size=32, shuffle=True,  collate_fn=collator)
    val_loader   = DataLoader(imdb_val_ds,   batch_size=32, shuffle=False, collate_fn=collator)

    t0=time.time()
    best = (-1, -1)  # (f1, acc)
    for e in range(1, epochs+1):
        model.train()
        for b in train_loader:
            b = {k: v.to(device) for k,v in b.items()}
            out = model(**b)
            loss = out.loss
            opt.zero_grad(); loss.backward(); opt.step()
        acc, f1 = eval_loader(model, val_loader)
        best = max(best, (f1, acc))
    return best[0], best[1], (time.time()-t0)/60, model  # f1, acc, minutes, model

grid = [
    (1e-4, 3e-5, 0.0), (1e-4, 5e-5, 0.0),
    (2e-4, 3e-5, 0.01), (2e-4, 5e-5, 0.01)
]
best_rec = None
for head_lr, bb_lr, wd in grid:
    f1v, accv, mins, _ = run_ft(head_lr, bb_lr, wd, epochs=3)
    print(f"FT trial head_lr={head_lr} bb_lr={bb_lr} wd={wd} -> val_f1={f1v:.4f} val_acc={accv:.4f} in {mins:.2f}m")
    if not best_rec or f1v > best_rec[0]:
        best_rec = (f1v, accv, head_lr, bb_lr, wd)

# retrain best on full train (train+val) and eval on test
# 1) Merge train+val for the final fit
imdb_full_train = concatenate_datasets([imdb_train_ds, imdb_val_ds])

final_collator = DataCollatorWithPadding(tokenizer=tok)
final_loader   = torch.utils.data.DataLoader(
    imdb_full_train, batch_size=32, shuffle=True, num_workers=0, collate_fn=final_collator
)

test_loader = torch.utils.data.DataLoader(
    imdb_tok["test"], batch_size=32, shuffle=False, num_workers=0, collate_fn=final_collator
)

# 2) Fresh model, freeze all → unfreeze last transformer layer + head
best_head_lr, best_bb_lr, best_wd = best_rec[2], best_rec[3], best_rec[4]

final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2).to(device)
for p in final_model.distilbert.parameters(): 
    p.requires_grad = False
for p in final_model.distilbert.transformer.layer[-1].parameters(): 
    p.requires_grad = True

head_params       = list(final_model.pre_classifier.parameters()) + list(final_model.classifier.parameters())
last_layer_params = list(final_model.distilbert.transformer.layer[-1].parameters())

opt = AdamW([
    {"params": head_params,       "lr": best_head_lr, "weight_decay": best_wd},
    {"params": last_layer_params, "lr": best_bb_lr,   "weight_decay": best_wd},
])

# 3) Time the final training
EPOCHS = 3
t0 = time.time()
for e in range(EPOCHS):
    final_model.train()
    for b in final_loader:
        b = {k: v.to(device) for k, v in b.items()}
        out = final_model(**b)
        loss = out.loss
        opt.zero_grad(); loss.backward(); opt.step()
train_time_min = (time.time() - t0) / 60.0

# 4) Test evaluation
acc_test, f1_test = eval_loader(final_model, test_loader)
print(f"[DistilBERT FT] test_acc={acc_test:.4f} test_f1={f1_test:.4f} | train_time_min={train_time_min:.2f}")

# 5) Log with timing
log_result(
    hub="Hugging Face",
    model="DistilBERT-base-uncased",
    task="IMDb",
    f1=f1_test,
    acc=acc_test,
    train_min=train_time_min,
    preprocessing=PREPROC_DESC_HF_NLP,
    notes=f"fine-tune: last layer; head_lr={best_head_lr}, bb_lr={best_bb_lr}, wd={best_wd}"
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FT trial head_lr=0.0001 bb_lr=3e-05 wd=0.0 -> val_f1=0.8656 val_acc=0.8656 in 5.73m


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FT trial head_lr=0.0001 bb_lr=5e-05 wd=0.0 -> val_f1=0.8687 val_acc=0.8688 in 5.73m


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FT trial head_lr=0.0002 bb_lr=3e-05 wd=0.01 -> val_f1=0.8688 val_acc=0.8688 in 5.65m
FT trial head_lr=0.0002 bb_lr=5e-05 wd=0.01 -> val_f1=0.8656 val_acc=0.8656 in 5.63m


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[DistilBERT FT] test_acc=0.8674 test_f1=0.8673 | train_time_min=5.77


#### ResNet18 — light fine-tune + tiny search. Search space: unfreeze: layer4 only; head LR: {1e-3, 3e-4}; backbone LR: {1e-4, 5e-5}; weight decay: {0, 1e-4}; epochs: 3

In [45]:
device = torch.device("mps") if torch.backends.mps.is_available() else \
         torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

def eval_cv(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            ps.append(logits.argmax(1).cpu().numpy())
            ys.append(yb.cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p, average="macro")

def run_resnet_ft(head_lr, bb_lr, wd, epochs=3):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 10)

    # freeze all
    for p in model.parameters(): p.requires_grad = False
    # unfreeze head + layer4
    for p in model.fc.parameters(): p.requires_grad = True
    for p in model.layer4.parameters(): p.requires_grad = True

    model = model.to(device)
    params = [
        {"params": model.fc.parameters(), "lr": head_lr, "weight_decay": wd},
        {"params": model.layer4.parameters(), "lr": bb_lr, "weight_decay": wd},
    ]
    opt = AdamW(params)

    t0=time.time()
    for e in range(1, epochs+1):
        model.train()
        for xb, yb in train_loader_resnet_s:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            loss = nn.functional.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
    acc_v, f1_v = eval_cv(model, val_loader_resnet)
    return f1_v, acc_v, (time.time()-t0)/60, model

grid = [
    (1e-3, 1e-4, 0.0), (1e-3, 5e-5, 1e-4),
    (3e-4, 1e-4, 0.0), (3e-4, 5e-5, 1e-4),
]
best = None
for head_lr, bb_lr, wd in grid:
    f1v, accv, mins, _ = run_resnet_ft(head_lr, bb_lr, wd, epochs=3)
    print(f"ResNet FT head_lr={head_lr} bb_lr={bb_lr} wd={wd} -> val_f1={f1v:.4f} val_acc={accv:.4f}")
    if not best or f1v > best[0]:
        best = (f1v, accv, head_lr, bb_lr, wd)

# retrain best on full train (= train_split + val)
# --- Final ResNet18 training on full train (train_split + val) WITH TIMING ---
# 1) Build full-train loader (train_split + val)
full_loader_resnet = DataLoader(
    ConcatDataset([cifar_train_resnet_split, cifar_val_resnet]),
    batch_size=64, shuffle=True, num_workers=0
)

# 2) Recreate model with best hyperparams, freeze all → unfreeze layer4 + head
head_lr, bb_lr, wd = best[2], best[3], best[4]

model_best = resnet18(weights=ResNet18_Weights.DEFAULT)
in_features = model_best.fc.in_features
model_best.fc = nn.Linear(in_features, 10)

for p in model_best.parameters(): 
    p.requires_grad = False
for p in model_best.layer4.parameters(): 
    p.requires_grad = True
for p in model_best.fc.parameters(): 
    p.requires_grad = True

model_best = model_best.to(device)

opt = AdamW(
    [
        {"params": model_best.fc.parameters(),     "lr": head_lr, "weight_decay": wd},
        {"params": model_best.layer4.parameters(), "lr": bb_lr,   "weight_decay": wd},
    ]
)

# 3) Time the final 3-epoch training on full train
EPOCHS = 3
t0 = time.time()
for e in range(EPOCHS):
    model_best.train()
    for xb, yb in full_loader_resnet:
        xb = xb.to(device); yb = yb.to(device)
        logits = model_best(xb)
        loss = nn.functional.cross_entropy(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
train_time_min_r = (time.time() - t0) / 60.0

# 4) Evaluate on test and log
acc_test, f1_test = eval_cv(model_best, test_loader_resnet)
print(f"[ResNet18 FT] test_acc={acc_test:.4f} test_f1={f1_test:.4f} | train_time_min={train_time_min_r:.2f}")

log_result(
    hub="PyTorch Hub",
    model="ResNet18",
    task="CIFAR-10",
    f1=f1_test,
    acc=acc_test,
    train_min=train_time_min_r,
    preprocessing=PREPROC_DESC_RESNET,
    notes=f"fine-tune: layer4; head_lr={head_lr}, bb_lr={bb_lr}, wd={wd}"
)

ResNet FT head_lr=0.001 bb_lr=0.0001 wd=0.0 -> val_f1=0.8916 val_acc=0.8922
ResNet FT head_lr=0.001 bb_lr=5e-05 wd=0.0001 -> val_f1=0.8895 val_acc=0.8898
ResNet FT head_lr=0.0003 bb_lr=0.0001 wd=0.0 -> val_f1=0.8983 val_acc=0.8990
ResNet FT head_lr=0.0003 bb_lr=5e-05 wd=0.0001 -> val_f1=0.8976 val_acc=0.8982
[ResNet18 FT] test_acc=0.8980 test_f1=0.8977 | train_time_min=3.96


### TF-Hub MobileNetV2 (tf_keras) — light fine-tune + tiny search. Search space: feat.trainable: {False, True}; LR: if frozen {1e-3, 5e-4}; if trainable {1e-5} (very small); epochs: 3

In [47]:
# --- safer MBv2 fine-tune sweep (tf_keras) ---
# If you want a smaller batch when trainable=True:
BATCH_FROZEN = 64
BATCH_TRAIN  = 16                           # <-- smaller when unfreezing
IMG_SIZE     = 224
FV_URL       = "https://tfhub.dev/google/imagenet/mobilenet_v2_130_224/feature_vector/5"

# Rebuild datasets at a given batch size (no aug, matches earlier preprocessing)
def make_batched(ds_raw, batch):
    return ds_raw.unbatch().batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds_frozen = make_batched(train_ds, BATCH_FROZEN)
val_ds_frozen   = make_batched(val_ds,   BATCH_FROZEN)
test_ds_batched = make_batched(test_ds,  BATCH_FROZEN)

train_ds_trainable = make_batched(train_ds, BATCH_TRAIN)
val_ds_trainable   = make_batched(val_ds,   BATCH_TRAIN)

def build_model(trainable=False, lr=1e-3):
    # release previous graphs/memory
    keras.backend.clear_session()
    gc.collect()

    feat = hub.KerasLayer(
        FV_URL,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        trainable=trainable,
        name=f"hub_fv_{'train' if trainable else 'frozen'}"
    )
    x_in = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x    = feat(x_in, training=False)       # <-- keep BN in inference mode
    y    = keras.layers.Dense(10, activation="softmax")(x)
    m    = keras.Model(x_in, y)
    m.compile(
        optimizer=keras.optimizers.legacy.Adam(learning_rate=lr),  # <-- legacy Adam
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

def eval_tf(model, ds):
    ys, ps = [], []
    for xb, yb in ds:
        yhat = model(xb, training=False).numpy()  # avoids predict() worker pool
        ps.append(np.argmax(yhat, axis=1))
        ys.append(yb.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return accuracy_score(y,p), f1_score(y,p, average="macro")

# Keep the sweep tiny; start frozen, then one cautious trainable run
configs = [
    {"trainable": False, "lr": 1e-3, "train_ds": train_ds_frozen,   "val_ds": val_ds_frozen},
    {"trainable": False, "lr": 5e-4, "train_ds": train_ds_frozen,   "val_ds": val_ds_frozen},
    {"trainable": True,  "lr": 1e-5, "train_ds": train_ds_trainable,"val_ds": val_ds_trainable},  # smaller batch
]

best = None  # will hold dict: {"f1":..., "acc":..., "cfg": {...}}

for cfg in configs:
    m = build_model(trainable=cfg["trainable"], lr=cfg["lr"])
    m.fit(cfg["train_ds"], validation_data=cfg["val_ds"], epochs=3,
          verbose=1, workers=1, use_multiprocessing=False)
    acc_v, f1_v = eval_tf(m, cfg["val_ds"])
    print(f"MBv2 FT trainable={cfg['trainable']} lr={cfg['lr']} -> val_f1={f1_v:.4f}")
    if (best is None) or (f1_v > best["f1"]):
        best = {"f1": f1_v, "acc": acc_v, "cfg": cfg}
    # cleanup current trial
    del m
    keras.backend.clear_session(); gc.collect()

# --- Final refit on FULL TRAIN (train + val) with BEST CONFIG, WITH TIMING ---
best_cfg = best["cfg"]
# choose full-train dataset matching the batch size of the best config
if best_cfg["trainable"]:
    full_train_ds = train_ds_trainable.concatenate(val_ds_trainable)
else:
    full_train_ds = train_ds_frozen.concatenate(val_ds_frozen)

final_model = build_model(trainable=best_cfg["trainable"], lr=best_cfg["lr"])

t0 = time.time()
final_model.fit(full_train_ds, epochs=3, verbose=1, workers=1, use_multiprocessing=False)
train_time_min_tf = (time.time() - t0) / 60.0

# --- Test evaluation & logging ---
acc_t, f1_t = eval_tf(final_model, test_ds_batched)
print(f"[MBv2 FT] test_acc={acc_t:.4f} test_f1={f1_t:.4f} | train_time_min={train_time_min_tf:.2f}")

log_result(
    hub="TF Hub",
    model="MobileNetV2-130 (feature_vector)",
    task="CIFAR-10",
    f1=f1_t, acc=acc_t,
    train_min=train_time_min_tf,
    preprocessing=PREPROC_DESC_TF_MBV2,
    notes=(f"fine-tune: trainable={best_cfg['trainable']}, lr={best_cfg['lr']}, "
           f"batch={BATCH_TRAIN if best_cfg['trainable'] else BATCH_FROZEN}, legacy Adam")
)

Epoch 1/3
    782/Unknown - 73s 88ms/step - loss: 0.5246 - accuracy: 0.8203

2025-09-12 02:56:59.475413: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-09-12 02:56:59.475437: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2025-09-12 02:56:59.475442: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 13129284643503927993
2025-09-12 02:56:59.475446: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 3847370526981778599
2025-09-12 02:56:59.475729: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 3820296112127365252
2025-09-12 02:56:59.475738: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 17830132570848719196
2025-09-1

782/782 [==============================] - 81s 99ms/step - loss: 0.5246 - accuracy: 0.8203 - val_loss: 0.3633 - val_accuracy: 0.8728
Epoch 2/3


2025-09-12 02:57:07.443141: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-09-12 02:57:07.443163: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2025-09-12 02:57:07.443171: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 2394278673834322067
2025-09-12 02:57:07.443174: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 15739413769924956889
2025-09-12 02:57:07.443177: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 2763531755476509501
2025-09-12 02:57:07.444649: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1134566601757015748


782/782 [==============================] - 78s 99ms/step - loss: 0.3739 - accuracy: 0.8706 - val_loss: 0.3413 - val_accuracy: 0.8802
Epoch 3/3
782/782 [==============================] - 78s 100ms/step - loss: 0.3382 - accuracy: 0.8823 - val_loss: 0.3224 - val_accuracy: 0.8916


...

2025-09-12 03:36:42.794463: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


MBv2 FT trainable=True lr=1e-05 -> val_f1=0.9325
Epoch 1/3
3437/3437 [==============================] - 706s 204ms/step - loss: 0.5398 - accuracy: 0.8438
Epoch 2/3


2025-09-12 03:48:30.277274: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-09-12 03:48:30.277296: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1827000022278123888
2025-09-12 03:48:30.277298: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 4083393278742737027
2025-09-12 03:48:30.277300: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 157833986454892559
2025-09-12 03:48:30.277376: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[Adam/add/_10]]


3437/3437 [==============================] - 704s 205ms/step - loss: 0.3639 - accuracy: 0.9035
Epoch 3/3
3437/3437 [==============================] - 712s 207ms/step - loss: 0.3053 - accuracy: 0.9236
[MBv2 FT] test_acc=0.9095 test_f1=0.9097 | train_time_min=35.37


2025-09-12 04:12:19.576202: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### TF-IDF + Logistic Regression — quick GridSearchCV

In [49]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf",   LogisticRegression(max_iter=1000, n_jobs=-1))
])

param_grid = {
    "tfidf__max_features": [20000, 40000, 80000],
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.5, 1.0, 2.0],
}

# Time the entire grid search (CV + internal refit)
t0 = time.time()
gs = GridSearchCV(pipe, param_grid, scoring="f1_macro", cv=3, n_jobs=-1, verbose=1, refit=True)
gs.fit(X_train, y_train)
search_time_min = (time.time() - t0) / 60.0

# Build a clean best pipeline and time the final refit-on-full-train (for fair comparison)
best = gs.best_params_
best_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=best["tfidf__max_features"],
                              ngram_range=best["tfidf__ngram_range"],
                              min_df=best["tfidf__min_df"])),
    ("clf",   LogisticRegression(max_iter=1000, n_jobs=-1, C=best["clf__C"]))
])

t1 = time.time()
best_pipe.fit(X_train, y_train)
refit_time_min = (time.time() - t1) / 60.0

# Test metrics
y_pred = best_pipe.predict(X_test)
f1_gs = f1_score(y_test, y_pred, average="macro")
acc_gs = accuracy_score(y_test, y_pred)
print(best, f"test_f1={f1_gs:.4f} test_acc={acc_gs:.4f}")
print(f"grid_search_min={search_time_min:.2f}  final_refit_min={refit_time_min:.2f}")

# Log: use final refit time for train_min (comparable to other models' final fits)
log_result(
    hub="Kaggle-style",
    model="TF-IDF + LogisticRegression (GS)",
    task="IMDb",
    f1=f1_gs,
    acc=acc_gs,
    train_min=refit_time_min,                # <-- comparable training time
    preprocessing=PREPROC_DESC_TFIDF,
    notes=f"grid: {best} | search_min={search_time_min:.2f}"
)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
{'clf__C': 2.0, 'tfidf__max_features': 20000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)} test_f1=0.8984 test_acc=0.8984
grid_search_min=1.00  final_refit_min=0.12


### Master Results Table

In [51]:
if "results" not in globals() or len(results) == 0:
    raise RuntimeError("No results found. Make sure you called log_result(...) at least once.")

def pick(d, *aliases):
    # return the first present key among aliases (case-insensitive)
    for a in aliases:
        if a in d: return d[a]
        for k in d.keys():
            if k.lower() == a.lower(): return d[k]
    return None

norm = []
for row in results:
    # handle objects like Series/Namespace
    if hasattr(row, "to_dict"): row = row.to_dict()
    d = dict(row)
    rec = {
        "Hub":            pick(d, "Hub","hub","source"),
        "Model":          pick(d, "Model","model","arch"),
        "Task":           pick(d, "Task","task","dataset"),
        "Params":         pick(d, "Params","params","param_count","n_params"),
        "TrainTime_min":  pick(d, "TrainTime_min","train_min","train_time_min","train_time_minutes"),
        "PeakVRAM_GB":    pick(d, "PeakVRAM_GB","vram_gb","peak_vram_gb","peak_mem_gb"),
        "F1":             pick(d, "F1","f1","f1_macro"),
        "Acc":            pick(d, "Acc","accuracy","acc"),
        "Prec":           pick(d, "Prec","precision","prec"),
        "Rec":            pick(d, "Rec","recall","rec"),
        "Latency_ms":     pick(d, "Latency_ms","latency_ms","latency"),
        "Preprocessing":  pick(d, "Preprocessing","preprocessing","preproc"),
        "Notes":          pick(d, "Notes","notes","phase","comment"),
    }
    norm.append(rec)

df = pd.DataFrame(norm)

# String & numeric coercion
for c in ["Hub","Model","Task","Preprocessing","Notes","Params"]:
    df[c] = df[c].astype("string").fillna("")
for c in ["TrainTime_min","PeakVRAM_GB","F1","Acc","Prec","Rec","Latency_ms"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Phase from Notes (robust to hyphen/space/case)
pat = re.compile(r"\b(baseline|linear[- ]probe|fine[- ]tune)\b", flags=re.I)
df["Phase"] = (
    df["Notes"].str.extract(pat, expand=False)
               .str.lower()
               .str.replace(" ", "-", regex=False)
).fillna("unspecified")

mask = (
    (df["Phase"] == "unspecified") &
    (
        df["Notes"].str.contains(r"\b(grid|tuned)\b", case=False, na=False) |
        df["Model"].str.contains(r"\(gs\)", case=False, na=False)
    )
)
df.loc[mask, "Phase"] = "fine-tune"

# Efficiency metric
df["F1_per_min"] = np.where(df["TrainTime_min"].gt(0), df["F1"]/df["TrainTime_min"], np.nan)

# Order & save
cols = ["Hub","Model","Task","Params","TrainTime_min","PeakVRAM_GB","F1","Acc","Prec","Rec","Latency_ms","Preprocessing","Notes","Phase","F1_per_min"]
df = df[cols].sort_values(["Task","Model","Phase","F1"], ascending=[True,True,True,False]).reset_index(drop=True)

display(df)
df.to_csv("all_runs_results.csv", index=False)
print("Saved -> all_runs_results.csv")

,Hub,Model,Task,Params,TrainTime_min,PeakVRAM_GB,F1,Acc,Prec,Rec,Latency_ms,Preprocessing,Notes,Phase,F1_per_min
0,TF Hub,MobileNetV2-130 (feature_vector),CIFAR-10,,35.367571,NaN,0.909721,0.90950,NaN,NaN,NaN,"Resize256→CenterCrop224; scale to [0,1]; featu...","fine-tune: trainable=True, lr=1e-05, batch=16,...",fine-tune,0.025722
1,TF Hub,MobileNetV2-130 (feature_vector),CIFAR-10,,4.186346,NaN,0.858679,0.85670,NaN,NaN,NaN,"Resize256→CenterCrop224; scale to [0,1]; featu...",linear-probe,linear-probe,0.205114
2,PyTorch Hub,ResNet18,CIFAR-10,,3.959921,NaN,0.897659,0.89800,NaN,NaN,NaN,Resize256→CenterCrop224; ToTensor; Normalize(I...,"fine-tune: layer4; head_lr=0.0003, bb_lr=0.000...",fine-tune,0.226686
3,PyTorch Hub,ResNet18,CIFAR-10,,4.653751,NaN,0.765713,0.76680,NaN,NaN,NaN,Resize256→CenterCrop224; ToTensor; Normalize(I...,linear-probe,linear-probe,0.164537
4,Hugging Face,DistilBERT-base-uncased,IMDb,,5.773927,NaN,0.867335,0.86744,NaN,NaN,NaN,Lowercase+HTML/punct cleanup; HF tokenizer (un...,"fine-tune: last layer; head_lr=0.0002, bb_lr=3...",fine-tune,0.150216
5,Hugging Face,DistilBERT-base-uncased,IMDb,,8.613636,NaN,0.816447,0.81652,NaN,NaN,NaN,Lowercase+HTML/punct cleanup; HF tokenizer (un...,linear-probe,linear-probe,0.094785
6,Kaggle-style,TF-IDF + LogisticRegression,IMDb,,NaN,NaN,0.896477,0.89648,NaN,NaN,NaN,Lowercase+HTML/punct cleanup; TF-IDF(max_featu...,baseline,baseline,NaN
7,Kaggle-style,TF-IDF + LogisticRegression (GS),IMDb,,0.116253,NaN,0.898358,0.89836,NaN,NaN,NaN,Lowercase+HTML/punct cleanup; TF-IDF(max_featu...,"grid: {'clf__C': 2.0, 'tfidf__max_features': 2...",fine-tune,7.727591


Saved -> all_runs_results.csv
